# Ekman Emotion Classification — Single-Stage E2E

**Pipeline:**
```
Text → [7-class multi-label] → anger / disgust / fear / joy / sadness / surprise / neutral
```

| | E2E |
|---|---|
| **Task** | Multi-label: 6 emotions + neutral (7 classes) |
| **Recommended** | DeBERTa-v3 |
| **Loss** | Three-tier Per-Class ASL |
| **Imbalance** | severe (joy/fear ≈ 10×, neutral ≈ 14k) → 3-tier strategy |

**Output structure:**
```
<run_base_dir>/run_e2e/
  deberta/                ← first run
    checkpoints/best.pth
    logs/training_log.csv
    logs/config.txt
    results/              ← report, metrics, charts
  deberta_2/              ← second run (no overwrite)
```

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/convitom/NLP_Emotion_Group_14.git

In [ ]:
!pip install torch transformers scikit-learn pandas numpy matplotlib tqdm pyyaml -q

In [ ]:
import os, sys
import torch

PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
cd /content/NLP_Emotion_Group_14/e2e

In [ ]:
from src.utils import load_config

CONFIG_PATH = 'config/config.yaml'
cfg = load_config(CONFIG_PATH)

print(f"run_base_dir : {cfg['run_base_dir']}")
print(f"Model        : {cfg['e2e']['model']['name']}")
print(f"Loss         : {cfg['e2e']['training']['loss']}")

## 1. Data Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(os.path.join(cfg['data']['data_dir'], cfg['data']['train_file']))
emotions = ['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise']
counts   = df[emotions].sum().sort_values(ascending=False)

n_emotion = (df[emotions].sum(axis=1) > 0).sum()
n_neutral = len(df) - n_emotion
median    = counts.median()

print(f'Total samples : {len(df):,}')
print(f'has_emotion   : {n_emotion:,} ({n_emotion/len(df)*100:.1f}%)')
print(f'neutral       : {n_neutral:,} ({n_neutral/len(df)*100:.1f}%)')
print(f'\nClass counts (all 7, sorted):')
all_counts = dict(counts)
all_counts['neutral'] = n_neutral
for name, c in sorted(all_counts.items(), key=lambda x: -x[1]):
    tier = 'very_rare' if c < median/3 else 'rare' if c < median else 'common'
    print(f'  {name:<12}: {int(c):>6}  [{tier}]')
print(f'\nmedian={median:.0f}  joy/fear ratio={counts["joy"]/counts["fear"]:.1f}×')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# All 7 classes bar chart
all_names = list(counts.index) + ['neutral']
all_vals  = list(counts.values) + [n_neutral]
colors7   = ['#c0392b' if v < median/3 else '#e67e22' if v < median else '#27ae60'
             if n != 'neutral' else '#3498db'
             for n, v in zip(all_names, all_vals)]

axes[0].bar(all_names, all_vals, color=colors7, edgecolor='white')
axes[0].axhline(median,   color='black', linestyle='--', label=f'median={median:.0f}')
axes[0].axhline(median/3, color='gray',  linestyle=':',  label=f'median/3={median/3:.0f}')
axes[0].set_title('7-Class Counts (red=very_rare, orange=rare, green=common, blue=neutral)')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)
for i, (n, v) in enumerate(zip(all_names, all_vals)):
    axes[0].text(i, v + 100, f'{int(v):,}', ha='center', fontsize=8)

# Pie: emotion vs neutral
axes[1].pie([n_emotion, n_neutral], labels=['has_emotion', 'neutral'],
            colors=['#e74c3c', '#3498db'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Emotion vs Neutral')

plt.tight_layout()
plt.show()

## 2. Training

**Task:** 7-class multi-label (6 emotions + neutral), ALL samples  
**Recommended model:** DeBERTa-v3  

**Three-tier treatment:**
| Tier | Classes | Aug copies | Sampler boost | pos_weight scale | ASL γ- |
|------|---------|-----------|--------------|-----------------|--------|
| very_rare | fear | ×4 | ×5 | ^2.0 | 0.5 |
| rare | disgust, neutral | ×2 | ×3 | ^1.5 | 1.0 |
| common | anger, joy, sadness, surprise | ×0 | ×1 | ^1.0 | 2.0 |

In [ ]:
import yaml

CONFIG_PATH = 'config/config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# -----------------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------------
cfg['run_base_dir'] = "/content/drive/MyDrive"  # đường dẫn lưu kết quả

# -----------------------------------------------------------------------------
# Data
# -----------------------------------------------------------------------------
cfg['data']['data_dir']   = "data/"
cfg['data']['train_file'] = "data1_train.csv"
cfg['data']['val_file']   = "data1_val.csv"
cfg['data']['test_file']  = "data1_test.csv"

cfg['data']['auto_split']  = False
cfg['data']['val_ratio']   = 0.10
cfg['data']['test_ratio']  = 0.10
cfg['data']['max_length']  = 128
cfg['data']['seed']        = 42
cfg['data']['num_workers'] = 2

# -----------------------------------------------------------------------------
# Model
# -----------------------------------------------------------------------------
cfg['e2e']['model']['name']    = "deberta"  # deberta / roberta / electra / bert
cfg['e2e']['model']['dropout'] = 0.2

# -----------------------------------------------------------------------------
# Training
# -----------------------------------------------------------------------------
cfg['e2e']['training']['epochs']                  = 20
cfg['e2e']['training']['batch_size']              = 32   # giảm xuống 16 nếu OOM
cfg['e2e']['training']['lr']                      = 2.0e-5
cfg['e2e']['training']['weight_decay']            = 0.01
cfg['e2e']['training']['optimizer']               = "adamw"
cfg['e2e']['training']['scheduler']               = "cosine_warmup"
cfg['e2e']['training']['warmup_ratio']            = 0.10
cfg['e2e']['training']['early_stopping_patience'] = 5
cfg['e2e']['training']['threshold']               = 0.5
cfg['e2e']['training']['loss']                    = "per_class_asl"

# -----------------------------------------------------------------------------
# Tier thresholds
# -----------------------------------------------------------------------------
cfg['e2e']['training']['very_rare_divisor'] = 3.0
cfg['e2e']['training']['rare_divisor']      = 1.0

# -----------------------------------------------------------------------------
# Synonym augmentation
# -----------------------------------------------------------------------------
cfg['e2e']['training']['augment_rare']         = True
cfg['e2e']['training']['aug_copies_very_rare'] = 4
cfg['e2e']['training']['aug_copies_rare']      = 2
cfg['e2e']['training']['aug_copies_common']    = 0

# -----------------------------------------------------------------------------
# Weighted sampler
# -----------------------------------------------------------------------------
cfg['e2e']['training']['use_weighted_sampler'] = True
cfg['e2e']['training']['sampler_power']        = 2.0
cfg['e2e']['training']['boost_very_rare']      = 5.0
cfg['e2e']['training']['boost_rare']           = 3.0
cfg['e2e']['training']['boost_common']         = 1.0

# -----------------------------------------------------------------------------
# pos_weight scale per tier
# -----------------------------------------------------------------------------
cfg['e2e']['training']['pw_scale_very_rare'] = 2.0
cfg['e2e']['training']['pw_scale_rare']      = 1.5
cfg['e2e']['training']['pw_scale_common']    = 1.0

# -----------------------------------------------------------------------------
# Per-class ASL gammas
# -----------------------------------------------------------------------------
cfg['e2e']['training']['asl_gamma_pos_very_rare'] = 0.0
cfg['e2e']['training']['asl_gamma_neg_very_rare'] = 0.5
cfg['e2e']['training']['asl_clip_very_rare']      = 0.0

cfg['e2e']['training']['asl_gamma_pos_rare'] = 0.5
cfg['e2e']['training']['asl_gamma_neg_rare'] = 1.0
cfg['e2e']['training']['asl_clip_rare']      = 0.0

cfg['e2e']['training']['asl_gamma_pos_common'] = 1.0
cfg['e2e']['training']['asl_gamma_neg_common'] = 2.0
cfg['e2e']['training']['asl_clip_common']      = 0.0

# Save overrides back to yaml so train.py reads them
with open(CONFIG_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)

print(f"Model  : {cfg['e2e']['model']['name']}")
print(f"Epochs : {cfg['e2e']['training']['epochs']}")
print(f"Batch  : {cfg['e2e']['training']['batch_size']}")
print(f"Loss   : {cfg['e2e']['training']['loss']}")

In [ ]:
from src.train import train

result = train(config_path=CONFIG_PATH)

RUN_DIR = result['run_dir']
print(f'\nRun directory: {RUN_DIR}')
print(f'Best epoch   : {result["best_epoch"]}')
for k, v in result['best_metrics'].items():
    print(f'  {k}: {v:.4f}')

In [ ]:
log = pd.read_csv(result['log_path'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(log.epoch, log.train_loss, label='train', marker='o', ms=4)
ax1.plot(log.epoch, log.val_loss,   label='val',   marker='s', ms=4)
ax1.set_title('E2E — Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(log.epoch, log.val_micro_f1,    label='micro_F1',    marker='o', ms=4)
ax2.plot(log.epoch, log.val_macro_f1,    label='macro_F1',    marker='s', ms=4)
ax2.plot(log.epoch, log.val_weighted_f1, label='weighted_F1', marker='^', ms=4)
ax2.set_title('E2E — Val F1')
ax2.set_xlabel('Epoch')
ax2.set_ylim(0, 1.05)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Evaluate on Test Set

In [ ]:
from src.test import evaluate

metrics = evaluate(config_path=CONFIG_PATH, run_dir=RUN_DIR)

print(f'Micro  F1    : {metrics["micro_f1"]:.4f}')
print(f'Macro  F1    : {metrics["macro_f1"]:.4f}')
print(f'Weighted F1  : {metrics["weighted_f1"]:.4f}')
print(f'Hamming Loss : {metrics["hamming"]:.4f}')
print(f'Subset Acc   : {metrics["subset_accuracy"]:.4f}')

In [ ]:
pc = pd.read_csv(os.path.join(metrics['out_dir'], 'per_class.csv'))
pc = pc.set_index('class').sort_values('f1')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['f1', 'precision', 'recall']):
    colors = ['#c0392b' if v < 0.5 else '#e67e22' if v < 0.7 else '#27ae60'
              for v in pc[col]]
    bars = ax.barh(pc.index, pc[col], color=colors, edgecolor='white')
    ax.set_xlim(0, 1.1)
    ax.set_title(f'Per-class {col.upper()}')
    ax.axvline(pc[col].mean(), color='black', linestyle='--', alpha=0.7,
               label=f'Mean={pc[col].mean():.3f}')
    ax.legend(fontsize=8)
    for bar, v in zip(bars, pc[col]):
        ax.text(v+0.01, bar.get_y()+bar.get_height()/2,
                f'{v:.2f}', va='center', fontsize=8)

plt.suptitle('E2E — Per-Class Metrics (7 classes, test set)', fontsize=12)
plt.tight_layout()
plt.show()
display(pc)

In [ ]:
from IPython.display import Image, display as ipy_display

plots = [
    ('pr_curve.png',            'PR Curves'),
    ('confusion_aggregate.png', 'Aggregated Confusion Matrix'),
    ('confusion.png',           'Per-Class Confusion Matrices'),
    ('heatmap.png',             'Probability Heatmap'),
    ('thresholds.png',          'Optimal Thresholds'),
]

for fname, title in plots:
    path = os.path.join(metrics['out_dir'], fname)
    if os.path.isfile(path):
        print(f'\n─── {title} ───')
        ipy_display(Image(path))

## 4. Results Summary

In [ ]:
print('=' * 62)
print(f'  Run : {result["run_name"]}')
print(f'  Dir : {RUN_DIR}')
print('=' * 62)
print(f'\n E2E — 7 Classes ({cfg["e2e"]["model"]["name"]})')
print(f'   Micro F1     : {metrics["micro_f1"]:.4f}')
print(f'   Macro F1     : {metrics["macro_f1"]:.4f}')
print(f'   Weighted F1  : {metrics["weighted_f1"]:.4f}')
print(f'   Hamming Loss : {metrics["hamming"]:.4f}')
print(f'   Subset Acc   : {metrics["subset_accuracy"]:.4f}')
print('=' * 62)

print('\n Per-class F1:')
pc_sorted = pc.sort_values('f1', ascending=False)
for name, row in pc_sorted.iterrows():
    bar = '█' * int(row['f1'] * 20)
    print(f'  {name:<12}: {row["f1"]:.4f}  {bar}')

## 5. Inference on New Texts

In [ ]:
import torch
from src.train import build_model
from src.dataloader import BACKBONE_REGISTRY, CLASS_NAMES
from transformers import AutoTokenizer

model_name = cfg['e2e']['model']['name']
ckpt = torch.load(os.path.join(RUN_DIR, 'checkpoints', 'best.pth'),
                  map_location=device, weights_only=False)

model = build_model(cfg['e2e'], num_labels=ckpt['num_labels']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

tokenizer = AutoTokenizer.from_pretrained(BACKBONE_REGISTRY[model_name]['pretrained'])
best_ts   = metrics['best_thresholds']   # per-class thresholds từ val set
max_len   = cfg['data']['max_length']

def predict(text):
    enc = tokenizer(text, max_length=max_len, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        logits = model(enc['input_ids'].to(device),
                       enc['attention_mask'].to(device))
    probs = torch.sigmoid(logits).float().cpu().numpy()[0]
    preds = (probs >= best_ts).astype(int)
    detected = {CLASS_NAMES[i]: float(probs[i])
                for i in range(len(CLASS_NAMES)) if preds[i] == 1}
    if not detected:
        top = CLASS_NAMES[int(probs.argmax())]
        detected = {top: float(probs.max())}
    print(f'  [{text[:70]}]')
    for name, p in sorted(detected.items(), key=lambda x: -x[1]):
        print(f'    → {name:<12}: {p:.3f}  {"█"*int(p*20)}')
    print()

print('Model loaded. Ready for inference.')

In [ ]:
test_sentences = [
    "I am so happy today, everything went perfectly!",
    "This is absolutely disgusting, I can't believe they did that.",
    "I'm really scared about what might happen next.",
    "The weather is nice today.",
    "I feel so angry and disappointed at the same time.",
    "Wow, I did not expect that at all — completely shocked!",
    "I miss her so much, it still hurts.",
]

for s in test_sentences:
    predict(s)